# colab_08 — Astrocyte subset: colab_05 verdict application, sub-clustering, and substate definition

Carries the **astrocyte** half of the colab_05 glia subset forward — the parallel of colab_07 for the other lineage. colab_05 gave the astrocytes a full six-lever residual-batch adjudication and left per-cluster verdicts (**cl22 = ambient**, **cl33 = real_confounded (keep)**, **cl42 / cl45 = donor_effect**) but never defined astrocyte **substate** labels. This notebook (1) applies those colab_05 cluster verdicts to the ca. 94,271 astrocytes — dropping the ambient / donor-effect technical clusters *before* anything downstream, so a technical pocket cannot seed its own substate subcluster (the cl44 lesson from colab_07); (2) sub-clusters the remaining astrocytes on the scVI latent to expose substate structure; (3) defines pre-committed **reactive / DAA (GFAP-high) vs resting** substate labels that **eval #1** consumes; and (4) runs the same lighter per-subcluster integrity scan as colab_07. Runs on standard runtime + **high-RAM** (loads the ca. 6.5 GB glia object); no GPU — there is no model training here.

**Lifecycle:** this is the scaffold (no outputs). Phase-1 (Sonnet + user) then Phase-2 (cold-Opus) review before running; interp cells are written against observed values into the `_OUTPUT` copy after the run. **Two design points are deliberately provisional and flagged for Phase-1 review:** (a) the drop-vs-keep disposition of the colab_05 donor-effect clusters cl42 / cl45 (§4a — dropped by default, mirroring the cl44 precedent, but they are real astrocytes leaving the CPT substrate); and (b) the reactive-vs-resting signature gene lists (§6a) — the canonical-marker-argmax trap that collapsed microglia "DAM" in colab_07 is expected to recur for astrocyte DAA, so the lists are a first-pass hypothesis to be trimmed against the 6a dotplot on the first run, not a settled panel.

## 1 — Setup

### 1a — Mount Drive + clone/pull repo + install env

Same pattern as colab_04–07: mount Drive, clone-or-pull the repo, pin numpy first, install the integration env, and add an explicit `leidenalg` install (the §5 Leiden sub-clustering) in case it is not pinned in `requirements_integration.txt`. No GPU is requested — sub-clustering, scoring and DE are CPU work; the only heavy resource is RAM for the ca. 6.5 GB glia object.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

# Pin numpy first so pip picks numpy-1.x-compatible wheels (same rationale as colab_01-07).
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt
# Leiden sub-clustering (§5) needs leidenalg + python-igraph; ensure present even if unpinned.
!pip install leidenalg igraph

## 2 — Environment capture

### 2a — pip freeze + env JSON

Identical reproducibility snapshot to the earlier notebooks: full `pip freeze` plus a structured JSON of Python / OS / library versions and the repo commit, written under `outputs/software_versions/`. Methods sections need the exact versions present at run time, not the pin file.

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_08"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":   NOTEBOOK_ID,
    "date":          TODAY,
    "python_version": sys.version,
    "platform":      platform.platform(),
    "os_release":    platform.release(),
    "gpu":           _run(["nvidia-smi", "-L"]),
    "git_commit":    _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scanpy_version":   _ver("scanpy"),
    "anndata_version":  _ver("anndata"),
    "leidenalg_version":_ver("leidenalg"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Load the glia subset + carve out astrocytes

### 3a — Load `glia_subset_full.h5ad`, subset astrocytes, sanity-check latent + raw counts

Load the colab_05 glia hand-off object (astrocytes + microglia, 149,375 cells; carries raw counts in `.X`, the scVI latent `X_scVI`, the original `leiden_scvi` clusters, and `cell_type`). Fail loud unless those are present and `.X` is integer counts. Then carve out the astrocytes (`cell_type == "astrocyte"`, ca. 94,271 cells) — colab_05 promoted cl32 to astrocyte and split perivascular macrophages (old cl0) into their own `cell_type`, so both are already handled by the `cell_type` field. The colab_05-flagged astrocyte clusters (**22 / 33 / 42 / 45**) are still present in this subset and are adjudicated in §4a before sub-clustering. Print the astrocyte cluster × study table, donor count, and APOE breakdown so the working set is characterized before anything is done to it.

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1
FIG_DIR = os.path.join(REPO_PATH, "figures", "colab_08")
os.makedirs(FIG_DIR, exist_ok=True)

GLIA_PATH = os.path.join(DRIVE_ROOT, "glia_subset", "glia_subset_full.h5ad")
if not os.path.exists(GLIA_PATH):
    raise FileNotFoundError(f"missing colab_05 glia subset {GLIA_PATH}")
glia = sc.read_h5ad(GLIA_PATH)
print("loaded glia subset:", glia.shape)

assert "X_scVI" in glia.obsm, "X_scVI missing — colab_05 glia subset expected"
assert "leiden_scvi" in glia.obs.columns, "leiden_scvi missing — needed for cluster verdicts"
assert "cell_type" in glia.obs.columns, "cell_type missing — needed to carve astrocytes"

# raw-counts guard (sorted deposits hide non-integer tails in a head slice -> random sample)
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f})"

# carve out astrocytes (cl0 = perivascular_mac split off in colab_05; cl32 promoted TO astrocyte)
astro = glia[glia.obs["cell_type"] == "astrocyte"].copy()
astro.obs["leiden_scvi"] = astro.obs["leiden_scvi"].astype(str)
del glia; gc.collect()
print("astrocyte subset:", astro.shape)
print("\nastrocyte leiden_scvi x study (counts):")
print(pd.crosstab(astro.obs["leiden_scvi"], astro.obs["study_id"]))
print("\ndonors:", astro.obs["donor_id"].nunique(),
      "| apoe_carrier:", astro.obs["apoe_carrier"].value_counts(dropna=False).to_dict())
_ram("astrocyte subset")

## 4 — Apply colab_05 astrocyte cluster verdicts

### 4a — Confirm the colab_05 per-cluster verdicts on this subset (composition, ambient, DE)

colab_05's astrocyte adjudication already ran the heavy six-lever residual-batch battery and issued per-cluster verdicts. This cell does **not** re-adjudicate — it **re-confirms** that those verdicts still describe the clusters as they sit in this subset, then records the disposition that §5a will apply. For each colab_05-flagged cluster it reports:

- **cl22 = `ambient` → drop.** colab_05 found it Li-pure (0.963) with oligodendrocyte/myelin contamination — an ambient-RNA pocket, not an astrocyte substate. Dropped so it cannot seed a spurious "ambient" subcluster or leak into the reactive axis.
- **cl42 / cl45 = `donor_effect` → drop (provisional — Phase-1 decision).** Each is ca. 98% a single donor with a technical signature (heat-shock / high-depth). By the cl44 precedent (colab_07 §5a) a donor-pure technical pocket is dropped *before* sub-clustering so it cannot mint its own substate. The trade-off: these are genuine astrocytes leaving the CPT substrate — flip their entry in `ASTRO_CLUSTER_VERDICTS` to `keep` if Phase-1 prefers to retain-and-flag instead.
- **cl33 = `real_confounded` → keep (flagged).** Single-donor but the only metabolic program that replicated cross-study in colab_05 — real biology. Kept, but it stays donor-confounded, so §6b re-flags it and eval #1/#2 must treat any cl33-driven signal as single-donor (unevaluable as a generalizable win).

Evidence assembled: per-cluster composition (dominant donor / study fraction, median counts, ambient proxy) for **all** astrocyte clusters with the flagged four highlighted; a lineage-marker dotplot (astro vs oligo / microglia / neuron) so cl22's off-lineage ambient band is visible; and Wilcoxon DE of each flagged cluster vs the rest (astro-subset only — within the project's compute budget, never on the full object). The dispositions are set in `ASTRO_CLUSTER_VERDICTS`; §5a applies the drops.

In [ ]:
# log-normalized layer for scoring/DE (.X stays raw counts) — rebuilt on the astro subset
astro.layers["lognorm"] = astro.X.copy()
sc.pp.normalize_total(astro, target_sum=1e4, layer="lognorm")
astro.uns.pop("log1p", None)
sc.pp.log1p(astro, layer="lognorm")
astro.uns.pop("log1p", None)

# colab_05 per-cluster verdicts (this cell re-CONFIRMS them, it does not re-adjudicate).
# default disposition applied in §5a: drop = removed before sub-clustering; keep = retained.
COLAB05_FLAGGED = {
    "22": {"colab05_verdict": "ambient",         "disposition": "drop",
           "why": "Li-pure (0.963) oligo/myelin ambient contamination, not an astro substate"},
    "42": {"colab05_verdict": "donor_effect",    "disposition": "drop",
           "why": "~98% single donor, heat-shock technical signature (cl44 precedent: drop before subclustering)"},
    "45": {"colab05_verdict": "donor_effect",    "disposition": "drop",
           "why": "~98% single donor, high-depth technical signature (cl44 precedent: drop before subclustering)"},
    "33": {"colab05_verdict": "real_confounded", "disposition": "keep",
           "why": "single-donor but only metabolic program replicating cross-study — real biology; kept + flagged"},
}
present_flagged = [cl for cl in COLAB05_FLAGGED if cl in set(astro.obs["leiden_scvi"])]
missing_flagged = [cl for cl in COLAB05_FLAGGED if cl not in present_flagged]
if missing_flagged:
    print(f"[verdicts] colab_05-flagged clusters absent from this subset (already handled upstream?): {missing_flagged}")

# ambient / off-lineage proxy: oligo + microglia + neuron markers (astro should be low on all three)
AMBIENT = [g for g in ["MBP","PLP1","MOBP","CSF1R","P2RY12","AIF1","SNAP25","RBFOX3"]
           if g in astro.var_names]
X_raw = astro.X
astro.X = astro.layers["lognorm"]
try:
    sc.tl.score_genes(astro, gene_list=AMBIENT, score_name="score_ambient", use_raw=False)
finally:
    astro.X = X_raw

# per-cluster composition for ALL astro clusters (flagged four highlighted below)
rows = []
for cl, idx in astro.obs.groupby("leiden_scvi", observed=True).groups.items():
    o = astro.obs.loc[idx]
    rows.append({
        "cluster": str(cl), "n": len(o),
        "top_donor_frac": round(float(o["donor_id"].value_counts(normalize=True).iloc[0]), 3),
        "top_study_frac": round(float(o["study_id"].value_counts(normalize=True).iloc[0]), 3),
        "median_counts": int(o["total_counts"].median()) if "total_counts" in o.columns else -1,
        "mean_ambient": round(float(o["score_ambient"].mean()), 3),
    })
comp = pd.DataFrame(rows).set_index("cluster")
print("per-astro-cluster composition (all clusters):")
with pd.option_context("display.width", 160):
    print(comp.sort_index(key=lambda s: s.astype(int)))
print("\ncolab_05-flagged clusters (verdict re-confirmation):")
print(comp.loc[present_flagged])
for cl in present_flagged:
    print(f"  cl{cl}: colab_05={COLAB05_FLAGGED[cl]['colab05_verdict']:15s} -> {COLAB05_FLAGGED[cl]['disposition']:4s} | {COLAB05_FLAGGED[cl]['why']}")

# lineage-marker dotplot: astro vs oligo/micro/neuron across all astro clusters
# (cl22 ambient should show an off-lineage oligo/myelin band)
DOT_GENES = ["AQP4","SLC1A2","GJA1","SLC1A3","GFAP",
             "MBP","PLP1","MOBP","CSF1R","P2RY12","AIF1","SNAP25","RBFOX3"]
dot_present = [g for g in DOT_GENES if g in astro.var_names]
X_raw = astro.X
astro.X = astro.layers["lognorm"]
try:
    sc.pl.dotplot(astro, dot_present, groupby="leiden_scvi", standard_scale="var", show=False)
    plt.savefig(os.path.join(FIG_DIR, "4a_astrocyte_cluster_marker_dotplot.png"),
                dpi=150, bbox_inches="tight")
    plt.show()  # render inline AND keep the saved PNG
finally:
    astro.X = X_raw

# DE of each flagged cluster vs the rest of the astrocytes (astro-subset Wilcoxon — within budget)
X_raw = astro.X
astro.X = astro.layers["lognorm"]
try:
    sc.tl.rank_genes_groups(astro, "leiden_scvi", groups=present_flagged, reference="rest",
                            method="wilcoxon", key_added="flagged_vs_rest")
finally:
    astro.X = X_raw
print("\ntop-12 up genes per flagged cluster (vs rest of astrocytes):")
for cl in present_flagged:
    print(f"  cl{cl}:", pd.DataFrame(astro.uns["flagged_vs_rest"]["names"])[cl].head(12).tolist())

# DISPOSITION applied in §5a. Drop clusters cannot seed a spurious substate subcluster (cl44 lesson).
# Flip any entry to "keep" here (Phase-1 decision for the cl42/cl45 donor-effect clusters).
ASTRO_CLUSTER_VERDICTS = {cl: COLAB05_FLAGGED[cl]["disposition"] for cl in present_flagged}
print("\n[verdicts] disposition applied in §5a:", ASTRO_CLUSTER_VERDICTS)

## 5 — Astrocyte sub-clustering

### 5a — Apply the verdict drops, then Neighbors + Leiden on `X_scVI` within astrocytes + UMAP + per-subcluster DE

The global 46-cluster solution split the astrocytes only coarsely (into the {22,28,30,31,32,33,42,45} block), so within-astrocyte substate structure is not resolved at that resolution. Re-running the neighbor graph and Leiden **on the astrocytes only** — on the same scVI latent used throughout (consistent with how colab_05/07 worked each lineage) — lets finer substates separate. A within-astrocyte re-embedding (fresh HVGs / PCA) is the fallback if substates do not resolve on the integration latent.

**Verdicts applied first.** The §4a dispositions (drop cl22 ambient + cl42/cl45 donor-effect; keep cl33 flagged) are applied at the top of this cell, *before* sub-clustering, so a technical / ambient pocket cannot seed its own substate subcluster — the same cl44 precedent from colab_07 §5a. Flip an entry in `ASTRO_CLUSTER_VERDICTS` to `keep` to retain it.

**Why the resolution matters here:** §6a assigns one substate label *per subcluster* (the mean reactive/resting z-score axis over each Leiden group), so the partition is load-bearing — too coarse dilutes a real reactive pocket into "intermediate" by averaging it with resting neighbours; too fine fragments one substate into noisy groups with unstable means. We keep cluster-level assignment (for its denoising) at a primary `resolution=1.0`, and report an ARI-stability check at res 0.5 / 1.5 so the post-run interp can judge how sensitive the partition is. The resolution-independent alternative — thresholding each cell's continuous score directly — was declined here (same as colab_07) because it forfeits that denoising and is noisier at single-cell level.

The UMAPs are coloured by subcluster, study (to spot any subcluster that is one study only), and APOE. Per-subcluster Wilcoxon DE gives each subcluster its top markers for the substate read in §6.

In [ ]:
# apply the §4a verdicts BEFORE sub-clustering, so an ambient/technical pocket cannot seed its own
# substate subcluster (cl44 precedent, colab_07 §5a)
drop_clusters = [cl for cl, v in ASTRO_CLUSTER_VERDICTS.items() if v == "drop"]
if drop_clusters:
    _before = astro.n_obs
    astro = astro[~astro.obs["leiden_scvi"].isin(drop_clusters)].copy()
    print(f"[verdicts] dropped clusters {drop_clusters}: {_before - astro.n_obs} cells; astrocytes now {astro.n_obs}")
else:
    print("[verdicts] no clusters marked drop — full astrocyte set retained")
kept_flagged = [cl for cl, v in ASTRO_CLUSTER_VERDICTS.items() if v == "keep"]
if kept_flagged:
    print(f"[verdicts] kept + flagged (still confounded, re-flagged in §6b): {kept_flagged}")

# sub-cluster astrocytes on the scVI latent (consistent with colab_05/07 use of X_scVI)
sc.pp.neighbors(astro, use_rep="X_scVI", n_neighbors=15, random_state=0)
sc.tl.leiden(astro, resolution=1.0, key_added="leiden_astro", random_state=0)
sc.tl.umap(astro, random_state=0)
print("astro subclusters:", astro.obs["leiden_astro"].nunique())
print(astro.obs["leiden_astro"].value_counts().sort_index())

# resolution-stability sanity-check: §6a assigns ONE substate label per subcluster (mean score),
# so the Leiden resolution is load-bearing. Report subcluster counts + ARI vs the primary res=1.0
# at neighbouring resolutions so the _OUTPUT interp can judge how stable the partition is.
from sklearn.metrics import adjusted_rand_score
print("\nresolution stability (vs res=1.0):")
for r in (0.5, 1.5):
    sc.tl.leiden(astro, resolution=r, key_added=f"_leiden_r{r}", random_state=0)
    ari = adjusted_rand_score(astro.obs["leiden_astro"], astro.obs[f"_leiden_r{r}"])
    print(f"  res={r}: {astro.obs[f'_leiden_r{r}'].nunique()} subclusters, ARI vs res=1.0 = {ari:.3f}")
    del astro.obs[f"_leiden_r{r}"]

print("\nleiden_astro x study (row-normalized):")
print(pd.crosstab(astro.obs["leiden_astro"], astro.obs["study_id"], normalize="index").round(3))

# per-subcluster marker DE (astro-subset Wilcoxon — within budget per the project rule)
X_raw = astro.X
astro.X = astro.layers["lognorm"]
try:
    sc.tl.rank_genes_groups(astro, "leiden_astro", method="wilcoxon", key_added="astro_de")
finally:
    astro.X = X_raw
print("\ntop-8 markers per astro subcluster:")
print(pd.DataFrame(astro.uns["astro_de"]["names"]).head(8))

# UMAPs
for color in ["leiden_astro", "study_id", "apoe_carrier"]:
    loc = "on data" if color == "leiden_astro" else "right margin"
    sc.pl.umap(astro, color=color, show=False, legend_loc=loc)
    plt.savefig(os.path.join(FIG_DIR, f"5a_umap_{color}.png"), dpi=150, bbox_inches="tight")
    plt.show()  # render inline AND keep the saved PNG
_ram("after subclustering")

## 6 — Substate definition + integrity scan

### 6a — Astrocyte substate axis: reactive / DAA (GFAP-high) vs resting (eval #1 label source)

This closes the eval #1 OPEN item for astrocytes, using the **rebuilt** method colab_07 arrived at after the microglia DAM-argmax collapse — because the same trap is expected here. Canonical DAA (disease-associated astrocytes; Habib et al. 2020) was defined in mouse, and its signature mixes (i) **pan-astrocytic genes** that are high-and-flat in *every* astrocyte (AQP4, SLC1A3, ALDH1L1) with (ii) **mouse-specific or dropout genes** (Serpina3n has no clean 1:1 human ortholog at the count level; several DAA markers sit at dropout in this intersection snRNA set). An absolute `score_genes` argmax over that list would again produce an "is-an-astrocyte" score, not a substate axis.

**The rebuilt axis (provisional gene lists — Phase-1 to validate against the dotplot).** The measurable reactive program here is **GFAP-high / DAA-like** (GFAP, CD44, VIM, CHI3L1, SERPINA3 — human reactive-astrocyte markers that should genuinely vary across subclusters) versus **resting** (SLC1A2, GLUL, GJA1, NRXN1 — homeostatic glutamate-handling / gap-junction genes that go *down* with reactivity and are well-detected). Three method choices, identical to colab_07: (1) trim to well-measured, discriminating genes only — drop pan-astrocytic inflators (AQP4, SLC1A3, ALDH1L1) and mouse/dropout genes; (2) **z-score each gene across all astrocytes** before pooling, so a constitutively-high gene cannot dominate its pole; (3) one **continuous axis** = mean(z, reactive) − mean(z, resting), with a band around zero for `intermediate`, instead of argmax of two non-comparable absolute scores. Labelled **"reactive," not "DAA,"** to avoid claiming a mouse template the data cannot support. Scored on raw-expression-derived lognorm, independent of any FM embedding, so the eval #1 probe against these labels stays non-circular.

**Expected to need a first-run rebuild.** The gene lists above are a hypothesis; on the first run the 6a signature dotplot must be read exactly as in colab_07 — any "reactive" gene that is high-and-flat across all subclusters is a pan-astrocytic inflator to move into `EXCLUDED_GENES`, and any listed gene sitting at dropout should be dropped. Do not trust the 3-way split until the dotplot confirms the panel discriminates.

**Four things carried into the interp (parallel to colab_07):**

- **eval #1 must be donor-held-out.** If the reactive pole concentrates in donor/study-confounded subclusters (see §6b), a probe "win" on a non-donor-held-out split could be recovering donor/study, not substate. eval #1 already inherits the global donor-level 70/15/15 split (contract §Eval #1), and the per-subcluster `confounded` flag is written into the committed JSON.
- **`INT_MARGIN = 0.10` is an unvalidated placeholder in z-score units** — calibrate once against the observed per-subcluster axis distribution post-run, same refine-at-pilot pattern as the contract.
- **Nuisance axes the donor/study check misses.** colab_07 found sex (Y-linked) and heat-shock subclusters slipping the donor/study threshold; cl42/cl45 (heat-shock / high-depth) are already dropped here, but the first run must still watch for sex / stress-driven astro subclusters and re-bucket the consequential (reactive-pole) ones via `NUISANCE_OVERRIDE` — populated *after* the first run, not pre-guessed.
- **"reactive" is not one uniform program.** Distinct astrocyte reactive states — e.g. the metallothionein / oxidative-stress program (MT1G/MT2A/MT1E) and the complement / A1-like program (C3) — are **not** the same as the GFAP-high axis and are recorded as limitations / candidate separate labels (`REACTIVE_BUCKET_NOTES`, filled post-run), not folded into "reactive."

The signature lists, the excluded genes (with reason), the per-subcluster assignment + confound flags, any nuisance-override record, and the eval-design requirements are written to `outputs/astrocyte_substate_signatures.json` as the committed definition.

In [ ]:
# Astrocyte substate axis (eval #1 label source) — built with the colab_07 rebuilt method from the
# start, because the microglia DAM failure mode is expected to recur for astrocyte DAA: canonical
# (mouse) DAA mixes pan-astrocytic genes (AQP4/SLC1A3/ALDH1L1, high-and-flat in ALL astrocytes) with
# mouse-only / dropout genes, so an absolute score_genes argmax would again be an "is-an-astrocyte"
# score. The measurable reactive program here is GFAP-high / DAA-like (GFAP/CD44/VIM/CHI3L1/SERPINA3,
# which should vary) vs resting (SLC1A2/GLUL/GJA1/NRXN1, well-detected, down with reactivity).
# Labeled "reactive" NOT "DAA" to avoid claiming a mouse template the data cannot support.
#
# Method (identical to colab_07 §6a): (1) trim to WELL-MEASURED, DISCRIMINATING genes; (2) z-score
# each gene across all astrocytes before pooling; (3) ONE continuous axis = mean(z,reactive) -
# mean(z,resting), band around 0 = "intermediate".
#
# PROVISIONAL (Phase-1 / first-run): validate these lists against the 6a dotplot below. Any
# "reactive" gene high-and-flat everywhere is a pan-astrocytic inflator -> move to EXCLUDED_GENES;
# any listed gene at dropout -> drop. Do not trust the split until the dotplot confirms discrimination.
SUBSTATE_SIG = {
    "reactive": ["GFAP", "CD44", "VIM", "CHI3L1", "SERPINA3"],   # GFAP-high / DAA-like reactive
    "resting":  ["SLC1A2", "GLUL", "GJA1", "NRXN1"],             # homeostatic glutamate-handling
}
EXCLUDED_GENES = {  # written into the committed JSON so the exclusion is auditable
    "pan_astrocytic_inflators": ["AQP4", "SLC1A3", "ALDH1L1"],
    "mouse_only_or_dropout":    ["SERPINA3N", "C4B", "GBP2", "OSMR"],
}

sig_present = {k: [g for g in gs if g in astro.var_names] for k, gs in SUBSTATE_SIG.items()}
for k, gs in sig_present.items():
    # fail loud rather than silently scoring an empty pole
    assert gs, f"[substate] {k}: zero genes present in var_names — cannot score this pole"
    miss = set(SUBSTATE_SIG[k]) - set(gs)
    if miss:
        print(f"[substate] {k}: absent from gene set -> {sorted(miss)}")

# z-score each signature gene across ALL astrocytes on the lognorm layer, then pool
all_sig = sig_present["reactive"] + sig_present["resting"]
L = astro.layers["lognorm"][:, [astro.var_names.get_loc(g) for g in all_sig]]
L = L.toarray() if sp.issparse(L) else np.asarray(L)
Z = (L - L.mean(0)) / (L.std(0) + 1e-9)
zdf = pd.DataFrame(Z, columns=all_sig)

astro.obs["score_reactive"] = zdf[sig_present["reactive"]].mean(axis=1).values
astro.obs["score_resting"]  = zdf[sig_present["resting"]].mean(axis=1).values
astro.obs["substate_axis"]  = astro.obs["score_reactive"] - astro.obs["score_resting"]

SUB = astro.obs.groupby("leiden_astro", observed=True)[
    ["score_reactive", "score_resting", "substate_axis"]].mean()
print("per-subcluster mean substate scores (z-units) + axis:")
print(SUB.round(3))

# provisional substate per subcluster from the continuous axis. INT_MARGIN is in z-score units,
# an UNVALIDATED placeholder: calibrate ONCE post-run against the observed per-subcluster axis
# distribution (recorded in the interp), same "refine at pilot" pattern as colab_07 / the contract.
INT_MARGIN = 0.10
def _call(a):
    return "intermediate" if abs(a) < INT_MARGIN else ("reactive" if a > 0 else "resting")
sub_assign_axis_only = {cl: _call(row["substate_axis"]) for cl, row in SUB.iterrows()}

# Manual nuisance override (same precedent as colab_07 cl10 / CL44_VERDICT — recorded, not silent).
# EMPTY until the first run: the §6b donor/study check does not catch every nuisance axis (colab_07
# found sex + heat-shock subclusters slipping it). cl42/cl45 (heat-shock/high-depth) are already
# dropped in §5a, but the first run must inspect the DE for any sex/stress-driven subcluster landing
# in the REACTIVE pole (the consequential bucket) and re-bucket it here — do NOT pre-guess entries.
NUISANCE_OVERRIDE = {}   # e.g. {"<cl>": {"to": "intermediate", "reason": "..."}} after the first run
sub_assign = dict(sub_assign_axis_only)
for cl_key, ov in NUISANCE_OVERRIDE.items():
    matches = [cl for cl in sub_assign if str(cl) == cl_key]
    assert matches, f"[nuisance override] subcluster {cl_key!r} not found in this run's clusters"
    for cl in matches:
        print(f"[nuisance override] cl{cl_key}: axis-only call "
              f"{sub_assign_axis_only[cl]!r} -> {ov['to']!r} ({ov['reason']})")
        sub_assign[cl] = ov["to"]

# Nuisance subclusters that slip the donor/study check but land in the majority/default (resting)
# bucket — documented, not overridden (lower stakes: they dilute rather than manufacture a finding).
# Fill from the first-run DE, same as colab_07's cl5/cl12.
KNOWN_NUISANCE_UNCONFOUNDED = {}

# Heterogeneity WITHIN the "reactive" call (qualitative, from DE + the 6a dotplot) so a reader does
# not treat the bucket as one uniform program. Distinct astro reactive states (metallothionein/
# oxidative-stress MT1G/MT2A/MT1E; complement/A1-like C3) are NOT the GFAP-high axis — record as
# limitations / candidate separate labels. No code/labeling effect. Fill from the first run.
REACTIVE_BUCKET_NOTES = {}

astro.obs["substate"] = astro.obs["leiden_astro"].map(sub_assign)
# fail loud on any unmapped subcluster instead of letting .astype(str) mint a silent "nan" label
assert not astro.obs["substate"].isna().any(), "subcluster(s) missing a substate assignment"
astro.obs["substate"] = astro.obs["substate"].astype(str)
print("\nprovisional substate per subcluster (axis-only):", sub_assign_axis_only)
print("provisional substate per subcluster (after nuisance override):", sub_assign)
print("substate cell counts:", astro.obs["substate"].value_counts().to_dict())

# eval#1 leakage guard: record per-subcluster donor/study dominance next to each substate call so
# the interp can see whether the reactive pole is donor/study-confounded. eval#1 already inherits the
# global donor-held-out split (contract), but a reactive substate that is single-donor in the held-out
# set is unevaluable, not a win.
dom = []
for cl, idx in astro.obs.groupby("leiden_astro", observed=True).groups.items():
    o = astro.obs.loc[idx]
    td = float(o["donor_id"].value_counts(normalize=True).iloc[0])
    ts = float(o["study_id"].value_counts(normalize=True).iloc[0])
    dom.append({"subcluster": str(cl), "n": int(len(o)),
                "substate": sub_assign[cl], "axis_only_call": sub_assign_axis_only[cl],
                "axis": round(float(SUB.loc[cl, "substate_axis"]), 3),
                "top_donor_frac": round(td, 3), "top_study_frac": round(ts, 3),
                "confounded": bool(td >= 0.50 or ts >= 0.60)})
dom_df = pd.DataFrame(dom).set_index("subcluster")
print("\nper-subcluster substate + confound flags:")
print(dom_df)

# committed signature definition (eval#1 label source)
SIG_PATH = os.path.join(REPO_PATH, "outputs", "astrocyte_substate_signatures.json")
with open(SIG_PATH, "w") as f:
    json.dump({"date": TODAY, "lineage": "astrocyte",
               "axis": "reactive_DAA_GFAP_high_vs_resting",
               "scoring": "per-gene z-score across all astrocytes (lognorm); pole = mean z; "
                          "substate_axis = reactive - resting; cluster-level mean",
               "signatures": SUBSTATE_SIG, "present": sig_present,
               "excluded_genes": EXCLUDED_GENES,
               "intermediate_margin_zunits": INT_MARGIN,
               "per_subcluster": dom,
               "per_subcluster_assignment": sub_assign,
               "per_subcluster_assignment_axis_only": sub_assign_axis_only,
               "colab05_cluster_verdicts": ASTRO_CLUSTER_VERDICTS,
               "nuisance_override": NUISANCE_OVERRIDE,
               "known_nuisance_unconfounded": KNOWN_NUISANCE_UNCONFOUNDED,
               "reactive_bucket_heterogeneity_notes": REACTIVE_BUCKET_NOTES,
               "eval1_requirements": {
                   "labels": ["reactive", "resting", "intermediate"],
                   "donor_held_out": True,
                   "reason": "reactive pole may concentrate in donor/study-confounded subclusters "
                             "(see per_subcluster confounded flag); a non-donor-held-out probe could "
                             "recover donor/study, not substate"},
               "uncaptured_programs": {
                   "metallothionein_oxidative_stress": ["MT1G", "MT2A", "MT1E"],
                   "complement_A1_like": ["C3"],
                   "note": "distinct astrocyte reactive states not scored by the GFAP-high axis"},
               "note": "expression-signature substate labels for the eval#1 probe; independent of FM "
                       "embeddings (not circular). Built with the colab_07 rebuilt method (z-scored "
                       "continuous axis) from the start; canonical mouse-DAA not directly recoverable. "
                       "Gene lists PROVISIONAL pending first-run dotplot validation. colab_05 ambient "
                       "(cl22) + donor-effect (cl42/cl45) clusters dropped before subclustering."},
              f, indent=2)
print("saved substate definition ->", SIG_PATH)

# substate-axis UMAP (colors come from .obs — no .X swap needed); rendered inline AND saved
sc.pl.umap(astro, color=["score_reactive", "score_resting", "substate_axis", "substate"],
           show=False)
plt.savefig(os.path.join(FIG_DIR, "6a_substate_scores_umap.png"), dpi=150, bbox_inches="tight")
plt.show()  # render inline AND keep the saved PNG

# signature dotplot — used genes + the excluded context genes so the exclusion is visible
X_raw = astro.X
astro.X = astro.layers["lognorm"]
try:
    context = [g for g in (EXCLUDED_GENES["pan_astrocytic_inflators"]
                           + EXCLUDED_GENES["mouse_only_or_dropout"]) if g in astro.var_names]
    dot_genes = sig_present["reactive"] + sig_present["resting"] + context
    sc.pl.dotplot(astro, dot_genes, groupby="leiden_astro", standard_scale="var", show=False)
    plt.savefig(os.path.join(FIG_DIR, "6a_substate_signature_dotplot.png"), dpi=150, bbox_inches="tight")
    plt.show()  # render inline AND keep the saved PNG
finally:
    astro.X = X_raw

### 6b — Per-subcluster integrity scan (donor / study / ambient)

The same lighter scan as colab_07 §6b. colab_05 already ran the heavy six-lever battery on the astrocytes (that is what produced the cluster verdicts applied in §4a), so here we **report** per-subcluster diagnostics on the *new* sub-clustering — dominant-donor fraction, dominant-study fraction, median counts / mito, and an ambient/doublet proxy (oligo / microglia / neuron marker leakage) — without re-running a gated 2nd-scVI. A subcluster that is donor-dominated (≥ 50%) or study-dominated (≥ 60%) is flagged for the interp; because cl33 was kept as `real_confounded`, its descendant subcluster(s) are expected to re-flag here and that is by design (documented, not evaluated as a generalizable win). The gated 2nd-scVI is run **only** if a flagged subcluster also turns out to carry the APOE signal eval #2 depends on.

In [ ]:
# lighter integrity scan: per-subcluster donor/study dominance + ambient/quality proxies
X_raw = astro.X
if "pct_counts_mt" not in astro.obs.columns or "total_counts" not in astro.obs.columns:
    astro.var["mt"] = astro.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(astro, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)

# off-lineage ambient proxy for astrocytes: oligo + microglia + neuron markers
AMBIENT = [g for g in ["MBP","PLP1","MOBP","CSF1R","P2RY12","AIF1","SNAP25","RBFOX3"]
           if g in astro.var_names]
astro.X = astro.layers["lognorm"]
try:
    sc.tl.score_genes(astro, gene_list=AMBIENT, score_name="score_ambient", use_raw=False)
finally:
    astro.X = X_raw

rows = []
for cl, idx in astro.obs.groupby("leiden_astro", observed=True).groups.items():
    o = astro.obs.loc[idx]
    rows.append({
        "subcluster": cl, "n": len(o),
        "top_donor_frac": round(float(o["donor_id"].value_counts(normalize=True).iloc[0]), 3),
        "top_study_frac": round(float(o["study_id"].value_counts(normalize=True).iloc[0]), 3),
        "median_counts": int(o["total_counts"].median()),
        "median_pct_mt": round(float(o["pct_counts_mt"].median()), 2),
        "mean_ambient": round(float(o["score_ambient"].mean()), 3),
    })
scan = pd.DataFrame(rows).set_index("subcluster")
DONOR_FLAG, STUDY_FLAG = 0.50, 0.60
scan["flag"] = np.where(scan["top_donor_frac"] >= DONOR_FLAG, "donor",
                np.where(scan["top_study_frac"] >= STUDY_FLAG, "study", ""))
print("per-subcluster integrity scan:")
with pd.option_context("display.width", 160):
    print(scan)
flagged = scan[scan["flag"] != ""]
if len(flagged):
    print(f"\n[FLAG] {len(flagged)} subcluster(s) donor/study-dominated — report in interp; "
          f"gated 2nd-scVI ONLY if a flagged cluster also carries the eval#2 APOE signal:")
    print(flagged)
else:
    print("\nno subcluster donor/study-dominated above thresholds — astrocyte substrate clean")

## 7 — Save + handoff

### 7a — Save astrocyte subset with substate labels; audit trace; commit instructions

Save the astrocyte subset (raw counts in `.X` for the FM substrate, plus `leiden_astro`, `substate`, and the carried-over `apoe_carrier` / donor / study metadata) to Drive; the working `lognorm` layer is dropped before saving (rebuildable, keeps the file small). Append an `astrocyte_subset` trace to the cumulative `audit_report.json` and print the WSL commit commands. Committable = env freeze + audit JSON + the substate-signature JSON; the h5ad and figures are Drive- / gitignore-only.

In [ ]:
import shlex
# cluster dispositions (drop cl22/cl42/cl45, keep cl33) were already applied in §5a — no membership
# change is needed here; the verdicts are recorded in the audit trace below.

ASTRO_DIR = os.path.join(DRIVE_ROOT, "astro_subset")
os.makedirs(ASTRO_DIR, exist_ok=True)
if "lognorm" in astro.layers:
    del astro.layers["lognorm"]                       # rebuildable; .X stays raw counts
if sp.issparse(astro.X) and astro.X.getformat() != "csr":
    astro.X = sp.csr_matrix(astro.X)
ASTRO_PATH = os.path.join(ASTRO_DIR, "astro_subset.h5ad")
astro.write_h5ad(ASTRO_PATH)
print("saved astrocyte subset ->", ASTRO_PATH, f"({os.path.getsize(ASTRO_PATH)/1e9:.2f} GB)")

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)
report["astrocyte_subset"] = {
    "status": "computed",
    "date": TODAY,
    "n_astrocytes": int(astro.n_obs),
    "n_subclusters": int(astro.obs["leiden_astro"].nunique()),
    "subcluster_on": "X_scVI",
    "substate_axis": "reactive_DAA_GFAP_high_vs_resting",
    "substate_counts": astro.obs["substate"].value_counts().to_dict(),
    "substate_signatures_file": os.path.relpath(SIG_PATH, REPO_PATH),
    "colab05_cluster_verdicts": ASTRO_CLUSTER_VERDICTS,
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH, SIG_PATH)]
print("\n=== Commit + push (from WSL — Colab has no git creds) ===")
print("  cd /content/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /content/ad-glia-fm-prep && git commit -m "
      "'colab_08: astrocyte subset — colab_05 verdict application, subclustering, reactive-vs-resting substate'")
print("  cd /content/ad-glia-fm-prep && git push")

### Carried forward to the FM notebooks

- **Astrocyte subset** (`astro_subset.h5ad`, raw genes) — the astrocyte CPT substrate and the eval #1 / eval #2 astrocyte working set (colab_05 ambient cl22 + donor-effect cl42/cl45 dropped; cl33 kept + flagged).
- **Substate labels** (`substate`: reactive / resting / intermediate) + their committed signature definition (`astrocyte_substate_signatures.json`) — the eval #1 label source for astrocytes. The axis is **reactive / DAA (GFAP-high) vs resting** (not canonical mouse DAA — see §6a), gene lists validated against the 6a dotplot on the first run, and eval #1 uses the **global donor-held-out split** (reactive pole may be donor/study-confounded; requirement recorded in the JSON).
- **Cluster dispositions** — cl22/cl42/cl45 dropped, cl33 kept + flagged; recorded in the audit trace.
- **eval #1 label source now COMPLETE for both lineages** — microglia (activated/homeostatic, colab_07) + astrocyte (reactive/resting, colab_08). The eval #1 substate linear probe can now be built. **Next: the FM notebooks** — zero-shot embedding (Geneformer + scGPT) on the two subsets, then CPT + PEFT across the three regimes, evaluated against these committed substate labels and the APOE axis.